<a href="https://colab.research.google.com/github/diegofuentes51/LAB07/blob/main/LAB07_PY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ----------------------------------------------------------
# PARTE 1: LIBRERÍAS NECESARIAS Y CONFIGURACIÓN INICIAL
# ----------------------------------------------------------
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')  # Evita mensajes molestos de advertencia

# Estilo para los gráficos
sns.set(style="whitegrid")

In [16]:
# ----------------------------------------------------------
# PARTE 2: CARGA DEL DATASET DE CÁNCER DE MAMA
# ----------------------------------------------------------

# URL del dataset desde el repositorio UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

# Nombres de columnas según documentación
columnas = ['ID', 'Clump_Thickness','Uniformity_Cell_Size','Uniformity_Cell_Shape',
            'Marginal_Adhesion','Single_Epithelial_Cell_Size','Bare_Nuclei',
            'Bland_Chromatin','Normal_Nucleoli','Mitoses','Class']

# Cargamos los datos al DataFrame
df = pd.read_csv(url, names=columnas)

# Reemplazamos valores faltantes representados con '?' por NaN
df.replace('?', np.nan, inplace=True)

# Eliminamos filas con datos faltantes
df.dropna(inplace=True)

# Convertimos 'Bare_Nuclei' a entero
df['Bare_Nuclei'] = df['Bare_Nuclei'].astype(int)

# Convertimos la variable objetivo 'Class': 2 (benigno) → 0, 4 (maligno) → 1
df['Class'] = df['Class'].map({2: 0, 4: 1})

# Mostramos un resumen inicial de los datos
print("Dimensiones del dataset:", df.shape)
df.head()


Dimensiones del dataset: (683, 11)


,ID,Clump_Thickness,Uniformity_Cell_Size,Uniformity_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1,3,1,1,0
1,1002945,5,4,4,5,7,10,3,2,1,0
2,1015425,3,1,1,1,2,2,3,1,1,0
3,1016277,6,8,8,1,3,4,3,7,1,0
4,1017023,4,1,1,3,2,1,3,1,1,0


In [17]:
# ----------------------------------------------------------
# PARTE 3: FUNCIÓN PARA CALCULAR EL INFORMATION VALUE (IV)
# ----------------------------------------------------------

def calc_iv(data, target, bins=10):
    """
    Calcula el Information Value (IV) para cada variable predictora en el DataFrame.

    Parámetros:
        data (pd.DataFrame): DataFrame que contiene variables predictoras + variable objetivo.
        target (str): Nombre de la columna objetivo (binaria: 0 para "good", 1 para "bad").
        bins (int): Cantidad de bins para variables numéricas (binning con qcut).

    Retorna:
        pd.DataFrame: IV por variable ordenado de mayor a menor.
    """
    iv_dict = {}

    for col in data.columns:
        if col == target:
            continue  # saltamos la variable objetivo

        # Binning para variables numéricas
        if pd.api.types.is_numeric_dtype(data[col]):
            try:
                binned = pd.qcut(data[col], q=bins, duplicates='drop')
            except ValueError:
                binned = data[col]  # si hay pocos valores únicos, se deja sin binning
        else:
            binned = data[col]

        df_temp = pd.DataFrame({'x': binned, 'y': data[target]})
        grouped = df_temp.groupby('x')

        # Conteo de "buenos" y "malos"
        stats = grouped['y'].agg(['count', 'sum'])
        stats.columns = ['total', 'bad']
        stats['good'] = stats['total'] - stats['bad']

        # Distribuciones
        stats['dist_good'] = stats['good'] / stats['good'].sum()
        stats['dist_bad'] = stats['bad'] / stats['bad'].sum()

        # Weight of Evidence (WoE)
        stats['woe'] = np.log((stats['dist_good'] + 1e-6) / (stats['dist_bad'] + 1e-6))

        # IV = (dist_good - dist_bad) * WoE
        stats['iv'] = (stats['dist_good'] - stats['dist_bad']) * stats['woe']
        iv = stats['iv'].sum()

        iv_dict[col] = iv

    iv_df = pd.DataFrame.from_dict(iv_dict, orient='index', columns=['IV'])
    iv_df = iv_df.sort_values(by='IV', ascending=False)

    # Clasificación interpretativa
    def interpret_iv(iv):
        if iv < 0.02:
            return 'Muy débil'
        elif iv < 0.1:
            return 'Débil'
        elif iv < 0.3:
            return 'Moderado'
        elif iv < 0.5:
            return 'Fuerte'
        else:
            return 'Muy fuerte'

    iv_df['Interpretación'] = iv_df['IV'].apply(interpret_iv)
    return iv_df


In [18]:
# ----------------------------------------------------------
# PARTE 4: CÁLCULO DEL IV PARA TODAS LAS VARIABLES PREDICTORAS
# ----------------------------------------------------------

# Creamos un nuevo DataFrame que conserva la columna objetivo
df_iv = df.drop(columns='ID')  # solo eliminamos la columna irrelevante

# Calculamos el IV
iv_result = calc_iv(df_iv, target='Class')

# Mostramos los resultados
print("📊 Information Value por variable:")
display(iv_result)


📊 Information Value por variable:


,IV,Interpretación
Uniformity_Cell_Size,9.722666,Muy fuerte
Uniformity_Cell_Shape,6.704774,Muy fuerte
Bland_Chromatin,6.417414,Muy fuerte
Clump_Thickness,5.743087,Muy fuerte
Normal_Nucleoli,4.923690,Muy fuerte
Bare_Nuclei,4.673082,Muy fuerte
Single_Epithelial_Cell_Size,4.089891,Muy fuerte
Marginal_Adhesion,3.343685,Muy fuerte
Mitoses,0.720707,Muy fuerte
